Digital Business University of Applied Sciences

Data Science und Management (M. Sc.)

DAMI01 / DATA01 Data Analytics

Prof. Dr. Daniel Ambach

Julia Schmid (200022)

***
# Time Series Clustering
# Teil 2: Data Preparation
***


In [15]:
# Imports
import pandas as pd
import numpy as np
import webbrowser
import os
from ydata_profiling import ProfileReport
from IPython.display import display

from parameter import (
    INPUT_PATH,
    MAP_DESCRIPTION,
    MAP_MERCHANT_STATE,
    MAP_ERROR_CATEGORY
)
from funktionen import get_key_value

In [16]:
# Einstellung, sodass alle Spalten eines Datensatzes angezeigt werden
pd.set_option("display.max_columns", None)

In [17]:
# Daten laden
path_temp = INPUT_PATH / "data_acquisition.csv"
df = pd.read_csv(path_temp)

***
## Überblick


In [18]:
# Anzahl der Zeilen und Spalten ausgeben
print(f"Anzahl Zeilen: {df.shape[0]}")
print(f"Anzahl Spalten: {df.shape[1]}")

Anzahl Zeilen: 51115337
Anzahl Spalten: 39


In [19]:
# Zehn zufällige Einträge ausgeben
df.sample(n=10)

,id_transaction,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,description,id_cards,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web,id_user,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
43548361,21319377,2018-06-01 16:04:00,1488,5685,$15.70,Online Transaction,9932,ONLINE,NaN,NaN,5311,NaN,Department Stores,1684,Mastercard,Debit,5999258860780535,06/2024,737,YES,2,$16731,02/2020,2020,No,1488,34,66,1985,11,Female,6427 12th Boulevard,41.65,-81.43,$20155,$41091,$86837,695,6
36268248,18974385,2017-01-20 15:13:00,1758,2154,$0.10,Chip Transaction,14528,Meridian,MS,39301.0,5499,NaN,Miscellaneous Food Stores,2154,Visa,Debit,4631099146168683,12/2018,597,YES,1,$13725,03/2007,2014,No,1758,57,66,1963,1,Male,126 Valley Drive,32.32,-90.20,$11061,$22556,$43260,645,4
43431993,21282111,2018-05-24 18:15:00,327,5512,$36.38,Online Transaction,39021,ONLINE,NaN,NaN,4784,Bad Card Number,Tolls and Bridge Fees,5512,Visa,Debit,4711207147444250,07/2023,641,YES,2,$11528,11/2007,2013,No,327,44,67,1975,7,Male,7485 Seventh Street,44.41,-71.97,$18069,$36843,$52248,772,5
9645674,10486178,2011-12-20 10:59:00,1104,5766,$55.04,Swipe Transaction,81536,Pompano Beach,FL,33063.0,5310,NaN,Discount Stores,5766,Visa,Debit,4061173043608031,05/2021,265,YES,1,$16578,12/2000,2010,No,1104,91,66,1928,3,Male,120 Lafayette Boulevard,26.23,-80.13,$18266,$40141,$805,850,6
19658059,13660900,2013-12-01 15:31:00,1591,3967,$60.34,Swipe Transaction,54936,Oak Harbor,WA,98278.0,5912,NaN,Drug Stores and Pharmacies,2847,Visa,Credit,4654712521098398,03/2020,269,YES,2,$9800,05/2003,2008,No,1591,58,66,1961,6,Female,2200 Third Drive,48.28,-122.62,$16537,$33717,$58236,698,6
5773869,9264782,2011-03-12 12:13:00,1459,177,$2.36,Swipe Transaction,22938,Albuquerque,NM,87121.0,5812,Technical Glitch,Eating Places and Restaurants,177,Visa,Credit,4382384960777396,05/2020,622,NO,2,$7100,01/2008,2011,No,1459,53,65,1967,2,Male,9385 Birch Street,35.11,-106.62,$15385,$31369,$0,707,3
45299464,21884034,2018-09-29 08:09:00,1084,3064,$7.15,Chip Transaction,59935,Tacoma,WA,98404.0,5499,NaN,Miscellaneous Food Stores,3232,Amex,Credit,338539852038786,05/2017,921,YES,1,$20900,06/2002,2011,No,1084,36,66,1984,1,Male,152 West Drive,47.26,-122.45,$31518,$64262,$75415,775,4
43322912,21246945,2018-05-17 10:06:00,674,4786,$4.08,Chip Transaction,20519,Coronado,CA,92118.0,5942,NaN,Book Stores,5574,Mastercard,Debit,5615295761814830,02/2022,923,YES,1,$1399,11/2009,2009,No,674,63,66,1957,1,Female,13 Sixth Boulevard,32.73,-117.08,$13805,$28147,$25987,756,7
37956017,19517576,2017-05-16 07:21:00,332,5816,$66.58,Swipe Transaction,25875,Shelbyville,IN,46176.0,5912,NaN,Drug Stores and Pharmacies,253,Visa,Debit,4680771406185126,01/2013,913,YES,1,$7452,01/2010,2018,No,332,61,65,1958,7,Male,8794 Catherine Lane,39.52,-85.77,$18324,$37362,$21600,703,4
42399614,20949599,2018-03-15 09:12:00,1324,3314,$81.60,Chip Transaction,60569,Arcadia,FL,34266.0,5300,NaN,Wholesale Clubs,5889,Mastercard,Debit,5404924988382806,12/2023,898,YES,3,$8889,12/2007,2013,No,1324,45,61,1974,11,Female,535 Hill Boulevard,27.33,-82.54,$19065,$38872,$94970,663,4


In [20]:
# Datensatz-Info ausgeben
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51115337 entries, 0 to 51115336
Data columns (total 39 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   id_transaction         int64  
 1   date                   object 
 2   client_id              int64  
 3   card_id                int64  
 4   amount                 object 
 5   use_chip               object 
 6   merchant_id            int64  
 7   merchant_city          object 
 8   merchant_state         object 
 9   zip                    float64
 10  mcc                    int64  
 11  errors                 object 
 12  description            object 
 13  id_cards               int64  
 14  card_brand             object 
 15  card_type              object 
 16  card_number            int64  
 17  expires                object 
 18  cvv                    int64  
 19  has_chip               object 
 20  num_cards_issued       int64  
 21  credit_limit           object 
 22  acct_open_date  

***
## Phase 3: Datenvorbereitung
***
### **Datenqualität**

### Duplikate

In [21]:
# Anzahl der Duplikate bestimmen
df_duplicates = df[df.duplicated()]
print(f"Dieser Datensatz besitz {len(df_duplicates)} Duplikate.")

Dieser Datensatz besitz 0 Duplikate.


### Fehlende Werte

In [22]:
# Fehlende Werte ermitteln
count_nan = df.isna().sum()

# Prozentsatz der NaN-Werte berechnen
percent_nan = round((count_nan / len(df)) * 100, 2)

# Werte in einem DataFrame speichern
df_nan = pd.DataFrame({
    "Anzahl NaN": count_nan,
    "Prozent NaN": percent_nan
}).sort_values(by="Anzahl NaN", ascending=False)

# Variablen mit fehlenden Werten inkl. Anzahl und Prozentanteil ausgeben
df_nan[df_nan["Anzahl NaN"] > 0]

,Anzahl NaN,Prozent NaN
errors,50306417,98.42
zip,6281710,12.29
merchant_state,5931904,11.60


Fehlende Werte treten ausschließlich bei kategorischen Variablen auf. Diese werden daher durch die Kategorien *Unknown* oder *None* ersetzt.

Bei der Variable **errors** lassen sich fehlende Werte inhaltlich als „kein Fehler aufgetreten" deuten. Sie werden daher mit dem Wert *None* gefüllt.

Bei den Variablen **zip** und **merchant_state** gibt es hingegen keinen inhaltlichen Anhaltspunkt, welcher konkrete Wert hinter den fehlenden Einträgen stecken könnte. Diese werden daher mit *Unknown* gekennzeichnet.

In [23]:
# Fehlende Werte der Variable "errors" als "None" kennzeichnen
df["errors"] = df["errors"].fillna("None")

# Variable "zip" als String speichern und
# fehlende Werte als "Unknown" kennzeichnen
df["zip"] = (
    pd.to_numeric(df["zip"], errors="coerce")
    .astype("Int64")
    .astype("string"))
df["zip"] = df["zip"].fillna("Unknown")

# Fehlende Werte der Variable "merchant_state" als "Unknown" kennzeichnen
df["merchant_state"] = df["merchant_state"].fillna("Unknown")

### Vollständigkeit

In [24]:
gesamt_zellen = df.shape[0] * df.shape[1]
fehlende_zellen = df.isna().sum().sum()
vollstaendigkeit = (
    (gesamt_zellen - fehlende_zellen) / gesamt_zellen) * 100
print(f"Datenvollständigkeit: {vollstaendigkeit:.2f}%")

# Quelle: Ambach, D. data analytics master.
# Abgerufen am 10.03.2026 von
# https://github.com/dan-am/data analytics master/
# blob/main/3 data preparation/data validation.py

Datenvollständigkeit: 100.00%


### **Irrelevante Spalten löschen**

Eine Begründung, warum diese Spalten gelöscht werden, steht in der Datei **[01_Business_Data_Understanding](01_Business_Data_Understanding.ipynb)**.

In [25]:
df = df.drop(
    columns=[
        "merchant_id",
        "id_cards",
        "id_user",
        "card_number",
        "expires",
        "cvv",
        "card_on_dark_web",
        "mcc",
        "card_type",
        "has_chip",
        "acct_open_date",
        "address",
        "latitude",
        "longitude",
        "current_age",
        "retirement_age",
        "merchant_city",
        "use_chip",
    ]
)

### **Kategorische Variablen**

In [26]:
# Kategorische Variablen bestimmen
categorical_vars = [
    col for col in df
    if df[col].dtype == "object"
]

# Eindeutige Werte der kategorischen Variablen ausgeben
for i in categorical_vars:
    print(i)
    print(df[i].unique())
    print("")

date
['2010-01-01 00:01:00' '2010-01-01 00:02:00' '2010-01-01 00:05:00' ...
 '2019-10-31 23:57:00' '2019-10-31 23:58:00' '2019-10-31 23:59:00']

amount
['$-77.00' '$14.57' '$80.00' ... '$397.54' '$693.96' '$694.30']

merchant_state
['ND' 'IA' 'CA' 'IN' 'MD' 'NY' 'Unknown' 'TX' 'HI' 'PA' 'WI' 'GA' 'AL'
 'CT' 'WA' 'MA' 'CO' 'NJ' 'OK' 'MT' 'FL' 'AZ' 'KY' 'LA' 'IL' 'OH' 'MO'
 'MI' 'KS' 'NC' 'AR' 'TN' 'NM' 'SC' 'MN' 'NV' 'OR' 'VA' 'SD' 'WV' 'ME'
 'MS' 'RI' 'NH' 'DE' 'VT' 'Mexico' 'ID' 'NE' 'DC' 'UT' 'Vatican City' 'WY'
 'Dominican Republic' 'Canada' 'AK' 'Costa Rica' 'Germany' 'China'
 'United Kingdom' 'Estonia' 'Tuvalu' 'Taiwan' 'United Arab Emirates'
 'Lithuania' 'Netherlands' 'Japan' 'Greece' 'Vietnam' 'Haiti' 'Ireland'
 'Singapore' 'France' 'South Africa' 'Thailand' 'Italy' 'Denmark'
 'Jamaica' 'Benin' 'Belgium' 'Sierra Leone' 'Indonesia' 'Colombia'
 'Switzerland' 'Portugal' 'New Zealand' 'Jordan' 'Guatemala' 'Hong Kong'
 'Finland' 'Mongolia' 'Saudi Arabia' 'Philippines' 'Norway' 'Hunga

Die Variable **date** wird in das Datenformat Datum geändert und dabei wird ausschließlich der Tag ohne die Uhrzeitangabe (yyy-mm-dd) als separate Variable betrachtet.

Bei den Variablen **amount**, **credit_limit**,  **per_capita_income**,  **yearly_income** und **total_debt** wird das Dollarzeichen entfernt und als numerische Variable gespeichert.

Für die Variablen **description**, **errors**,  **merchant_state** werden Kategorien gebildet, um die Werte zu gruppieren und die Auswertung zu vereinfachen.


In [27]:
# Variable "date" als Datetime speichern
df["date"] = pd.to_datetime(df["date"])

# Tag aus dem Variable "date" extrahieren
df["day"] = df["date"].dt.date

In [28]:
# Dollarzeichen entfernen und als numerische Spalten speichern
list_dollar_change = [
    "amount",
    "credit_limit",
    "per_capita_income",
    "yearly_income",
    "total_debt",
]

for i in list_dollar_change:
    # Dollarzeichen entfernen
    df[i] = df[i].str[1:]
    # In numerische Spalte umwandeln
    df[i] = pd.to_numeric(df[i], errors="coerce")

In [29]:
# Kategorien für "description" erstellen
df["description_category"] = df["description"].apply(
    lambda x: get_key_value(x, MAP_DESCRIPTION)
)
df = df.drop(columns=["description"])
df["description_category"].value_counts()

description_category
Food_and_Beverage               19085160
Automotive                       9344898
Retail_General                   5460328
Travel_and_Transportation        3593642
Health_and_Medical               3402133
Financial_and_Legal              2404938
Electronics_and_Technology       1418134
Home_and_Garden                  1262886
Entertainment_and_Recreation     1073906
Books_and_Media                   933437
Utilities                         915159
Professional_Services             662748
Beauty_and_Personal_Care          559090
Metals_and_Manufacturing          486750
Clothing_and_Apparel              448529
Lodging                            63599
Name: count, dtype: int64

In [30]:
# Kategorien für "merchant_state" erstellen
df["merchant_category"] = df["merchant_state"].apply(
    lambda x: get_key_value(x, MAP_MERCHANT_STATE)
)
df = df.drop(columns=["merchant_state"])
df["merchant_category"].value_counts()

merchant_category
US_States        44833627
North_America      172617
Europe             111304
Asia                48696
South_America        9773
Africa               4099
Oceania              3317
Name: count, dtype: int64

In [31]:
# Kategorien für "errors" erstellen
df["error_category"] = df["errors"].apply(
    lambda x: get_key_value(x, MAP_ERROR_CATEGORY)
)
df = df.drop(columns=["errors"])
df["error_category"].value_counts()

error_category
No_Error           50306417
Single_Error         805395
Multiple_Errors        3525
Name: count, dtype: int64

### **Numerischen Variablen**

In [32]:
# Numerische Variablen ausgeben
numerical_vars = [
    col for col in df
    if df[col].dtype != "object"
]
for i in range(0, len(numerical_vars), 10):
    print(", ".join(numerical_vars[i:i + 10]))

id_transaction, date, client_id, card_id, amount, zip, num_cards_issued, credit_limit, year_pin_last_changed, birth_year
birth_month, per_capita_income, yearly_income, total_debt, credit_score, num_credit_cards


Die Variable **zip**, **birth_year**, **birth_month** werden als kategorische Variable gespeichert.

In [33]:
list_numeric_to_string = ["zip", "birth_year", "birth_month"]
for i in list_numeric_to_string:
    df[i] = df[i].astype(str)  # Umwandlung in String
    print(df[i].value_counts())  # Anzahl der Werte anzeigen

Bestimmung der Outlier

In [34]:
# Neue Bestimmung der numerischen Werte
# (wg. vorheriger Umwandlung in kategorische Variablen)
numerical_vars = [
    col for col in df
    if df[col].dtype != "object"
]

# Outlier mit der IQR-Methode bestimmen
for i in numerical_vars:
    var_q1 = df[i].quantile(0.25)
    var_g3 = df[i].quantile(0.75)
    var_iqr = var_g3 - var_q1
    # Anzahl der Outlier
    n_outlier = (
        (df[i] < var_q1 - 1.5 * var_iqr)
        | (df[i] > var_g3 + 1.5 * var_iqr)).sum()
    # Prozentanteil der Outlier
    p_outlier = round(n_outlier / len(df) * 100, 1)
    print(
        f"Variable {i} hat {n_outlier} Ausreißer, "
        f"was {p_outlier} Prozent entspricht"
    )

Variable id_transaction hat 0 Ausreißer, was 0.0 Prozent entspricht
Variable date hat 0 Ausreißer, was 0.0 Prozent entspricht
Variable client_id hat 0 Ausreißer, was 0.0 Prozent entspricht
Variable card_id hat 0 Ausreißer, was 0.0 Prozent entspricht
Variable amount hat 4092799 Ausreißer, was 8.0 Prozent entspricht
Variable num_cards_issued hat 0 Ausreißer, was 0.0 Prozent entspricht
Variable credit_limit hat 1781394 Ausreißer, was 3.5 Prozent entspricht
Variable year_pin_last_changed hat 130211 Ausreißer, was 0.3 Prozent entspricht
Variable per_capita_income hat 3086297 Ausreißer, was 6.0 Prozent entspricht
Variable yearly_income hat 2641763 Ausreißer, was 5.2 Prozent entspricht
Variable total_debt hat 833089 Ausreißer, was 1.6 Prozent entspricht
Variable credit_score hat 1303847 Ausreißer, was 2.6 Prozent entspricht
Variable num_credit_cards hat 216225 Ausreißer, was 0.4 Prozent entspricht


Vor dem Hintergrund, dass es sich um ein Clustering-Projekt handelt, werden die Ausreißer nicht weiter behandelt. Die Ausreißer in den betroffenen Variablen stellen reales Verhalten der Kunden dar. Eine Entfernung dieser Datenpunkte würde die Clustering-Ergebnisse verfälschen.

### **Cleaned-Datensatz**

In [35]:
print("Cleaned-Datensatz:", df.shape)
display(df.head())
df.info()

Cleaned-Datensatz: (51115337, 22)


,id_transaction,date,client_id,card_id,amount,zip,card_brand,num_cards_issued,credit_limit,year_pin_last_changed,birth_year,birth_month,gender,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,day,description_category,merchant_category,error_category
0,7475327,2010-01-01 00:01:00,1556,2972,-77.00,58523,Mastercard,2,24772,2010,1989,7,Female,23679,48277,110153,740,4,2010-01-01,Food_and_Beverage,US_States,No_Error
1,7475327,2010-01-01 00:01:00,1556,2972,-77.00,58523,Visa,2,15300,2020,1989,7,Female,23679,48277,110153,740,4,2010-01-01,Food_and_Beverage,US_States,No_Error
2,7475327,2010-01-01 00:01:00,1556,2972,-77.00,58523,Mastercard,2,55,2008,1989,7,Female,23679,48277,110153,740,4,2010-01-01,Food_and_Beverage,US_States,No_Error
3,7475327,2010-01-01 00:01:00,1556,2972,-77.00,58523,Amex,2,11200,2020,1989,7,Female,23679,48277,110153,740,4,2010-01-01,Food_and_Beverage,US_States,No_Error
4,7475328,2010-01-01 00:02:00,561,4575,14.57,52722,Mastercard,1,26960,2010,1971,6,Male,18076,36853,112139,834,5,2010-01-01,Retail_General,US_States,No_Error


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51115337 entries, 0 to 51115336
Data columns (total 22 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   id_transaction         int64         
 1   date                   datetime64[ns]
 2   client_id              int64         
 3   card_id                int64         
 4   amount                 float64       
 5   zip                    object        
 6   card_brand             object        
 7   num_cards_issued       int64         
 8   credit_limit           int64         
 9   year_pin_last_changed  int64         
 10  birth_year             object        
 11  birth_month            object        
 12  gender                 object        
 13  per_capita_income      int64         
 14  yearly_income          int64         
 15  total_debt             int64         
 16  credit_score           int64         
 17  num_credit_cards       int64         
 18  day                 

***
## Phase 4: Feature Engineering
***
Aus dem aufbereiteten Datensatz werden zwei Teildatensätze erstellt: eine Zeitreihe mit den Variablen card_id, day und amount, die als Grundlage für das Clustering dient, sowie ein ergänzender Datensatz mit den übrigen Variablen. Der Datensatz mit den ergänzenden Informationen wird im Anschluss herangezogen, um die gebildeten Cluster inhaltlich zu analysieren und die Ergebnisse zu bewerten.


### **Zeitreihen**

Für alle Karten-ID wird für jeden Tag ein Eintrag erstellt (vollständige Zeitreihe)


In [36]:
# Datensatz: Pro Karten-ID Summe der Umsätze (amount) pro Tag (day)
df_timeseries_pre = (
    df.groupby(["card_id", "day"])
    .agg(amount_sum=("amount", "sum"))
    .reset_index()
)

# Tag in ein Datetime-Format bringen
df_timeseries_pre["day"] = pd.to_datetime(df_timeseries_pre["day"])

# Liste aller Karten-IDs
list_cards = df_timeseries_pre["card_id"].unique()
# Liste aller Tage zwischen den Min-Tag und Max-Tag die im Datensatz vorkommen
list_days = pd.date_range(
    df_timeseries_pre["day"].min(),
    df_timeseries_pre["day"].max(),
)

# Vollständige Karte-Tag-Kombinationen erstellen
combination_card_day = pd.MultiIndex.from_product(
    [list_cards, list_days],
    names=["card_id", "day"]
)

# Erstellen eines Datensatz, bei dem jede Karte für jeden Tag
# den tatsächlichen Umsatz oder den Wert 0 hat
df_timeseries = (
    df_timeseries_pre.set_index(["card_id", "day"])
    .reindex(combination_card_day, fill_value=0)
    .reset_index()
)

df_timeseries.head()


,card_id,day,amount_sum
0,0,2010-01-01,653.68
1,0,2010-01-02,64.80
2,0,2010-01-03,450.76
3,0,2010-01-04,0.00
4,0,2010-01-05,122.84


In [37]:
df_timeseries.shape

(14618961, 3)

Die Karten-IDs, bei denen mehr als 50 Prozent der Tageswerte den Betrag 0 aufweisen, werden aus dem Datensatz entfernt, da sie zu wenig Transaktionsaktivität zeigen und somit nur eine geringe Aussagekraft für das Clustering besitzen.

In [38]:
# # Karten-IDs mit mehr als 50 % Null-Werten ermitteln und herausfiltern
df_timeseries_50 = (
    df_timeseries
    .assign(zero=df_timeseries["amount_sum"].eq(0))  # True/False für 0-Wert
    .groupby("card_id")
    .agg(
        days_total=("day", "nunique"),   # Anzahl Tage
        days_zero=("zero", "sum"),        # True zählt als 1
    )
    .assign(
        percent_zero=lambda x: 100 * x["days_zero"] / x["days_total"]
    )
    .sort_values("percent_zero", ascending=False)
)

card_ids_under_50 = (
    df_timeseries_50
    .loc[df_timeseries_50["percent_zero"] <= 50]
    .index
    .tolist()
)

df_timeseries = df_timeseries[
    df_timeseries["card_id"].isin(card_ids_under_50)
]

In [39]:
# Liste der Karten-IDs
list_card_ids = df_timeseries["card_id"].unique()

In [40]:
# Pivotisierung: jede Zeile repräsentiert eine Karte
# Fehlende Werte werden mit 0 gefüllt
df_timeseries = df_timeseries.pivot(
    index="card_id", columns="day", values="amount_sum"
).fillna(0)

In [41]:
df_timeseries.shape

(1786, 3591)

In [42]:
# Daten speichern
path_temp = INPUT_PATH / "data_timeseries.csv"
df_timeseries.to_csv(path_temp, index=True)

### **Zusatzinfos**
Ergänzend wird ein Zusatzinfo-Datensatz erstellt, der die übrigen Variablen pro Karte zusammenfasst. Dabei werden numerische Spalten durch ihren Median und kategoriale Spalten durch den jeweils häufigsten Wert aggregiert. Die Ergebnisse werden anschließend zu einem Datensatz zusammengeführt, sodass zu jede card_id eindeutig zusammengefassten Informationen vorliegen.

In [43]:
df_add_info = df.drop(["date", "id_transaction", "amount", "day"], axis=1)

# Nur Karten-IDs, die in der Zeitreihe sind
df_add_info = df_add_info[df_add_info["card_id"].isin(list_card_ids)]

In [44]:
# Numerische Spalten aggregieren (Median)
numerical_vars = df_add_info.select_dtypes(include="number").columns
df_num = df_add_info.groupby("card_id")[numerical_vars].median()

# Kategorische Spalten aggregieren (häufigster Wert)
categorical_vars = df_add_info.select_dtypes(exclude="number").columns
df_cat = df_add_info.groupby("card_id")[categorical_vars].agg(
    lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan
)

# Numerische und kategorische Ergebnisse zusammenführen
df_zusatzinfo = df_num.join(df_cat)

# card_id Spalte löschen, da diese Spalte als Index gespeichert ist
df_zusatzinfo = df_zusatzinfo.drop("card_id", axis=1)

# df_zusatzinfo.head()

**Hinweis**: Die Skalierung der Daten erfolgt im Skript **[03_Modeling_Evaluation](03_Modeling_Evaluation.ipynb)**., da dort jeweils 100 zufällig gewählte Zeitreihen in fünf Läufen gewählt werden. Die Skalierung erfolgt dann auf die ausgewählten Zeitreihen.

In [45]:
# Daten speichern
path_temp = INPUT_PATH / "data_add_info.csv"
df_zusatzinfo.to_csv(path_temp, index=True)

***
Profilingreport

In [46]:
# Profilingreport der Zusatzinfos laden
pr = ProfileReport(df_zusatzinfo, title="Financial Transactions Data")
filename_pr = "../output/add_info_data_pr.html"
path_pr = os.path.abspath(filename_pr)

pr.to_file(path_pr)  # ProfileReport als HTML speichern
webbrowser.open(f"file://{path_pr}")  # ProfileReport im Browser öffnen

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 207.26it/s]


True

***
***